In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Load dataset

In [3]:
with open("input.txt", "r", encoding='utf-8') as file:
    text = file.read()

length = len(text)
print("length of the dataset is ", length)

sample = text[0:500]
print(sample)

length of the dataset is  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


## Build vocabulary

In [4]:
chars = sorted(set(text))
vocab_string = "".join(chars)
vocab_size = len(chars)

# print(vocab_string)
# print(vocab_size)

In [5]:
stoi = {}
itos = {}

for i, ch in enumerate(chars):
    stoi[ch] = i
    itos[i] = ch

# print(stoi["a"])
# print(itos[39])
# print(stoi["\n"])

## Encode / decode

In [6]:
def encode(s):
    list1 = []
    for ch in s:
        list1.append(stoi[ch])
    return list1

def decode(i):
    list2 = []
    for no in i:
        list2.append(itos[no])
    return "".join(list2)

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


## Tokenize full dataset

In [7]:
data = torch.tensor(encode(text), dtype = torch.long)

print(data.shape)
print(data.dtype)
print(data[:100])

torch.Size([1115394])
torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


## Train / val split

In [8]:
n = int(90/100 * len(data))
train_data = data[:n]
val_data = data[n:]

print(train_data.shape)
print(val_data.shape)

torch.Size([1003854])
torch.Size([111540])


## Block size demo

In [9]:
block_size = 8

x = train_data[:8]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context.tolist()} then target is {target}")

when input is [18] then target is 47
when input is [18, 47] then target is 56
when input is [18, 47, 56] then target is 57
when input is [18, 47, 56, 57] then target is 58
when input is [18, 47, 56, 57, 58] then target is 1
when input is [18, 47, 56, 57, 58, 1] then target is 15
when input is [18, 47, 56, 57, 58, 1, 15] then target is 47
when input is [18, 47, 56, 57, 58, 1, 15, 47] then target is 58


## Batching

In [10]:
batch_size = 32

def get_batch(split):
    if split == "train":
        data = train_data
    else:
        data = val_data

    randomPosition = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in randomPosition])
    y = torch.stack([data[i+1 :i+block_size+1 ] for i in randomPosition])

    return x,y

In [11]:
xb, yb = get_batch("train")
print(xb.shape)
print(xb)
print(yb.shape)
print(yb)

torch.Size([32, 8])
tensor([[50, 47, 44, 43,  8,  0,  0, 13],
        [46, 47, 57,  1, 40, 56, 53, 61],
        [43,  8,  0,  0, 16, 33, 23, 17],
        [58, 43, 56, 10,  0, 15, 53, 51],
        [ 1, 54, 53, 61, 43, 56,  1, 53],
        [33, 30, 17, 26, 15, 17, 10,  0],
        [32, 46, 56, 43, 43,  7, 54, 47],
        [56, 52, 43, 63,  6,  1, 58, 53],
        [10,  0, 26, 53, 40, 50, 43,  1],
        [52,  5, 42,  1, 40, 63,  1, 63],
        [49, 52, 53, 61,  1, 57, 46, 43],
        [61, 52,  6,  1, 58, 46, 43, 52],
        [53, 59,  1, 61, 46, 39, 58,  1],
        [57,  1, 41, 56, 47, 51, 57, 53],
        [50, 47, 40, 50, 43,  0, 27, 44],
        [42, 63,  1, 58, 47, 51, 43, 10],
        [ 0, 13, 52, 42,  1, 49, 43, 43],
        [ 1, 47, 57,  1, 57, 58, 56, 39],
        [39, 60, 43, 52,  1, 51, 39, 63],
        [58, 46, 39, 58,  1, 51, 39, 49],
        [50,  5, 42,  6,  1, 58, 46, 63],
        [ 1, 53, 44, 44,  6,  0, 14, 43],
        [57,  1, 54, 59, 52, 47, 57, 46],
        [ 1, 2

## Bigram language model

In [12]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [13]:
@torch.no_grad()
def loss_estimator():
    out = {}
    m.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(200)
        for k in range(200):
            xb, yb = get_batch(split)
            logits, loss = m(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean()

    m.train()
    return out

## Train the model

In [14]:
m = BigramLanguageModel(vocab_size)

optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [15]:
for steps in range(10000):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if steps % 1000 == 0:
        losses = loss_estimator()
        print(f"steps {steps} training loss {losses['train']: .4f} validation loss {losses['val']: .4f}")

steps 0 training loss  4.6687 validation loss  4.6580
steps 1000 training loss  3.6987 validation loss  3.6947
steps 2000 training loss  3.1229 validation loss  3.1119
steps 3000 training loss  2.8054 validation loss  2.8017
steps 4000 training loss  2.6397 validation loss  2.6446
steps 5000 training loss  2.5607 validation loss  2.5680
steps 6000 training loss  2.5133 validation loss  2.5300
steps 7000 training loss  2.5001 validation loss  2.5111
steps 8000 training loss  2.4868 validation loss  2.4940
steps 9000 training loss  2.4741 validation loss  2.4940


In [16]:
torch.save(m.state_dict(), "bigram_model.pt")

## Generate sample text

In [17]:
start = torch.zeros((1, 1), dtype=torch.long)
output = m.generate(start, max_new_tokens=100)

print(decode(output[0].tolist()))


Whe e;
The'demuengl by I bll pthas! d ses momar:
My d sendedin:
BENI e:

Whanes
Shory fato hersthenm


## Toy example: bag-of-words averaging (self-attention prep)

In [ ]:
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, : t+1]
        xbow[b, t] = torch.mean(xprev, 0)

print(x[0])
print(xbow[0])

tensor([[-0.3101, -0.0629],
        [ 0.2634,  0.3291],
        [ 1.0008, -0.4787],
        [ 0.2782, -0.0406],
        [ 0.1874, -0.5566],
        [ 0.8873,  1.4786],
        [ 0.4400,  0.1450],
        [-1.9770, -0.5834]])
tensor([[-0.3101, -0.0629],
        [-0.0233,  0.1331],
        [ 0.3181, -0.0708],
        [ 0.3081, -0.0633],
        [ 0.2840, -0.1619],
        [ 0.3845,  0.1115],
        [ 0.3924,  0.1163],
        [ 0.0963,  0.0288]])


In [27]:
a = torch.tril(torch.ones(3, 3))
a = a / a.sum(1, keepdim=True)
c = torch.randn(3, 2)
d = a @ c

print(a)
print(c)
print(d)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[ 0.5748,  1.9060],
        [-0.1183,  0.4053],
        [-0.1048,  2.4545]])
tensor([[0.5748, 1.9060],
        [0.2283, 1.1556],
        [0.1172, 1.5886]])


In [31]:
wei = torch.tril(torch.ones(8, 8))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x
print(torch.allclose(xbow, xbow2))

True


In [33]:
tril = torch.tril(torch.ones(8, 8))
wei = torch.tril(torch.zeros(8, 8))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print(torch.allclose(xbow, xbow3))

True


In [38]:
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
v = value(x)

wei = q @ k.transpose(-2, -1)

wei = wei * head_size ** -0.5

tril = torch.tril(torch.ones(8, 8))

wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

out = wei @ v

print(out.shape)
print(wei[0])

torch.Size([4, 8, 16])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6928, 0.3072, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2354, 0.4972, 0.2674, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1634, 0.3171, 0.2236, 0.2960, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2539, 0.1485, 0.1750, 0.2944, 0.1282, 0.0000, 0.0000, 0.0000],
        [0.1151, 0.2539, 0.2012, 0.1326, 0.1484, 0.1488, 0.0000, 0.0000],
        [0.0974, 0.1180, 0.1537, 0.1764, 0.1340, 0.1602, 0.1603, 0.0000],
        [0.1411, 0.1113, 0.0757, 0.1197, 0.0948, 0.1477, 0.1226, 0.1871]],
       grad_fn=<SelectBackward0>)


In [40]:
n_embd = 32

head = Head(16)

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        tril = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("tril", tril)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)
        out = wei @ v
        return out

head = Head(16)
out = head(x)
print(out.shape)

torch.Size([4, 8, 16])
